# Trivima — SD + LoRA + CLIP Image Encoder

Replaces the weak latent_proj hack with a real CLIP ViT-L vision encoder.
The UNet now gets 257 semantic tokens describing the input photo, which is what SD's cross-attention layers were actually designed for.

**Run all:** Runtime → Run all

## 1. Install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q diffusers transformers peft accelerate datasets huggingface_hub pyarrow

## 2. Load SD 1.5 + LoRA + CLIP image encoder

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler
from peft import LoraConfig, get_peft_model
from transformers import CLIPVisionModel

device = 'cuda'

# VAE (frozen)
vae = AutoencoderKL.from_pretrained('stabilityai/sd-vae-ft-mse').to(device)
vae.requires_grad_(False); vae.eval()

# CLIP ViT-L vision encoder (frozen) — outputs (B, 257, 1024) tokens
clip_vision = CLIPVisionModel.from_pretrained('openai/clip-vit-large-patch14').to(device)
clip_vision.requires_grad_(False); clip_vision.eval()
CLIP_DIM = 1024

# UNet + LoRA
unet = UNet2DConditionModel.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5', subfolder='unet'
).to(device)
lora_config = LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.1,
    target_modules=['to_q', 'to_v', 'to_k', 'to_out.0'],
)
unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

# Project CLIP 1024 -> SD's expected 768
clip_proj = nn.Sequential(
    nn.LayerNorm(CLIP_DIM),
    nn.Linear(CLIP_DIM, 768),
).to(device)

class PoseEncoder(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(16, dim), nn.SiLU(), nn.Linear(dim, dim))
    def forward(self, p):
        return self.mlp(p.reshape(p.shape[0], -1))

pose_encoder = PoseEncoder(768).to(device)

scheduler = DDPMScheduler(num_train_timesteps=1000)
scheduler.alpha_cumprod = scheduler.alphas_cumprod.to(device)
print('SD + LoRA + CLIP ready')

## 3. Download dataset

In [ ]:
import os
from huggingface_hub import snapshot_download

os.makedirs('/content/data/re10k_hf', exist_ok=True)
snapshot_download(
    repo_id='Wouter01/re10k_pixelsplat_hard',
    repo_type='dataset',
    local_dir='/content/data/re10k_hf',
    allow_patterns=['data/train-0000[0-3]-of-00036.parquet'],
    max_workers=4,
)
!ls -lh /content/data/re10k_hf/data/

## 4. Dataset (returns CLIP-preprocessed input + SD-preprocessed input + target)

In [ ]:
import io, glob
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pyarrow.parquet as pq
import torchvision.transforms as T

# CLIP normalization (different from SD)
CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD  = [0.26862954, 0.26130258, 0.27577711]

class RE10KParquet(Dataset):
    def __init__(self, parquet_glob, sd_size=256, max_pairs=None):
        self.files = sorted(glob.glob(parquet_glob))
        self.index = []
        for fi, p in enumerate(self.files):
            n = pq.ParquetFile(p).metadata.num_rows
            for r in range(n):
                self.index.append((fi, r))
                if max_pairs and len(self.index) >= max_pairs: break
            if max_pairs and len(self.index) >= max_pairs: break
        self.tables = {}
        self.sd_tf = T.Compose([
            T.Resize((sd_size, sd_size), T.InterpolationMode.LANCZOS),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3),
        ])
        self.clip_tf = T.Compose([
            T.Resize((224, 224), T.InterpolationMode.BICUBIC),
            T.ToTensor(),
            T.Normalize(CLIP_MEAN, CLIP_STD),
        ])
        print(f'Dataset: {len(self.index)} pairs from {len(self.files)} shards')

    def _table(self, fi):
        if fi not in self.tables:
            self.tables[fi] = pq.read_table(self.files[fi])
        return self.tables[fi]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        fi, ri = self.index[i]
        t = self._table(fi)
        cond = t.column('conditioning_image')[ri].as_py()['bytes']
        gt   = t.column('ground_truth_image')[ri].as_py()['bytes']
        cond_img = Image.open(io.BytesIO(cond)).convert('RGB')
        gt_img   = Image.open(io.BytesIO(gt)).convert('RGB')
        return {
            'input_sd':   self.sd_tf(cond_img),
            'input_clip': self.clip_tf(cond_img),
            'target':     self.sd_tf(gt_img),
            'pose':       torch.eye(4),
        }

dataset = RE10KParquet('/content/data/re10k_hf/data/train-*.parquet', sd_size=256, max_pairs=10000)
loader = DataLoader(dataset, batch_size=24, shuffle=True, num_workers=4, drop_last=True, pin_memory=True)
print(f'Batches/epoch: {len(loader)}')

## 5. Train (CLIP cond + 10% drop for CFG, 20 epochs)

In [ ]:
import time, numpy as np

os.makedirs('/content/checkpoints_clip', exist_ok=True)
DRIVE_DIR = '/content/drive/MyDrive/trivima_checkpoints'
HAS_DRIVE = os.path.isdir('/content/drive/MyDrive')
if HAS_DRIVE:
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print('Drive available')
else:
    print('Drive NOT mounted — local only')

EPOCHS = 20
LR = 1e-4
COND_DROPOUT = 0.1  # for classifier-free guidance

trainable = list(unet.parameters()) + list(clip_proj.parameters()) + list(pose_encoder.parameters())
optimizer = torch.optim.AdamW([p for p in trainable if p.requires_grad], lr=LR, weight_decay=0.01)
best_loss = float('inf')

def save_ckpt(path, loss, epoch):
    torch.save({
        'unet': {k: v for k, v in unet.state_dict().items() if 'lora' in k.lower()},
        'clip_proj': clip_proj.state_dict(),
        'pose_encoder': pose_encoder.state_dict(),
        'loss': loss, 'epoch': epoch,
    }, path)

for epoch in range(EPOCHS):
    t0 = time.time()
    losses = []
    unet.train(); clip_proj.train(); pose_encoder.train()

    for bi, batch in enumerate(loader):
        inp_sd   = batch['input_sd'].to(device, non_blocking=True)
        inp_clip = batch['input_clip'].to(device, non_blocking=True)
        tgt      = batch['target'].to(device, non_blocking=True)
        pose     = batch['pose'].to(device, non_blocking=True)
        B = inp_sd.shape[0]

        with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
            tgt_lat = vae.encode(tgt).latent_dist.sample() * 0.18215
            clip_feats = clip_vision(pixel_values=inp_clip).last_hidden_state  # (B,257,1024)

        with torch.autocast('cuda', dtype=torch.bfloat16):
            cond_tokens = clip_proj(clip_feats)              # (B,257,768)
            pose_emb = pose_encoder(pose).unsqueeze(1)       # (B,1,768)
            enc_h = torch.cat([cond_tokens, pose_emb], dim=1)  # (B,258,768)

            # Classifier-free guidance dropout
            if COND_DROPOUT > 0:
                drop_mask = (torch.rand(B, device=device) < COND_DROPOUT).view(B, 1, 1)
                enc_h = torch.where(drop_mask, torch.zeros_like(enc_h), enc_h)

            t = torch.randint(0, 1000, (B,), device=device).long()
            noise = torch.randn_like(tgt_lat)
            noisy = scheduler.add_noise(tgt_lat, noise, t)

            pred = unet(noisy, t, encoder_hidden_states=enc_h).sample
            loss = F.mse_loss(pred.float(), noise.float())

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()

        losses.append(loss.item())
        if (bi+1) % 20 == 0:
            print(f'  [{epoch+1}/{EPOCHS}] {bi+1}/{len(loader)} loss={np.mean(losses[-20:]):.4f}')

    avg = float(np.mean(losses))
    dt = time.time() - t0
    print(f'Epoch {epoch+1}/{EPOCHS}: loss={avg:.4f} ({dt:.0f}s)')

    if avg < best_loss:
        best_loss = avg
        save_ckpt('/content/checkpoints_clip/best.pt', best_loss, epoch)
        if HAS_DRIVE:
            save_ckpt(f'{DRIVE_DIR}/best_clip.pt', best_loss, epoch)
        print(f'  -> saved: {best_loss:.4f}')

print(f'\nDone! Best loss: {best_loss:.4f}')

## 6. Generate with classifier-free guidance

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ckpt = torch.load('/content/checkpoints_clip/best.pt', map_location=device)
unet.load_state_dict(ckpt['unet'], strict=False)
clip_proj.load_state_dict(ckpt['clip_proj'])
pose_encoder.load_state_dict(ckpt['pose_encoder'])
unet.eval(); clip_proj.eval(); pose_encoder.eval()
print(f"Loaded, best loss: {ckpt['loss']:.4f}")

batch = next(iter(loader))
inp_sd   = batch['input_sd'][:1].to(device)
inp_clip = batch['input_clip'][:1].to(device)
gt       = batch['target'][:1].to(device)
pose     = batch['pose'][:1].to(device)

GUIDANCE = 5.0   # classifier-free guidance scale (1.0 = off, 7.5 typical)
STRENGTH = 0.6   # img2img strength

with torch.no_grad():
    inp_lat = vae.encode(inp_sd).latent_dist.sample() * 0.18215
    clip_feats = clip_vision(pixel_values=inp_clip).last_hidden_state
    cond_tokens = clip_proj(clip_feats)
    pose_emb = pose_encoder(pose).unsqueeze(1)
    enc_cond = torch.cat([cond_tokens, pose_emb], dim=1)
    enc_uncond = torch.zeros_like(enc_cond)

    n_steps = 50
    start_step = int(n_steps * STRENGTH)
    timesteps = torch.linspace(999, 0, n_steps, device=device).long()
    start_t = timesteps[n_steps - start_step]

    noise = torch.randn_like(inp_lat)
    latent = scheduler.add_noise(inp_lat, noise, start_t.unsqueeze(0))

    for idx in range(n_steps - start_step, n_steps):
        i = int(timesteps[idx])
        i_next = int(timesteps[idx + 1]) if idx + 1 < n_steps else 0
        t = torch.full((1,), i, device=device, dtype=torch.long)
        # CFG: two passes
        np_uncond = unet(latent, t, encoder_hidden_states=enc_uncond).sample
        np_cond   = unet(latent, t, encoder_hidden_states=enc_cond).sample
        noise_pred = np_uncond + GUIDANCE * (np_cond - np_uncond)

        a_t = scheduler.alpha_cumprod[i]
        a_next = scheduler.alpha_cumprod[i_next] if i_next > 0 else torch.tensor(1.0, device=device)
        pred_x0 = (latent - torch.sqrt(1 - a_t) * noise_pred) / torch.sqrt(a_t)
        pred_x0 = pred_x0.clamp(-4, 4)
        latent = torch.sqrt(a_next) * pred_x0 + torch.sqrt(1 - a_next) * noise_pred

    gen = vae.decode(latent / 0.18215).sample
    gen = (gen.clamp(-1, 1) + 1) / 2

def to_np(x):
    return x[0].cpu().permute(1, 2, 0).float().numpy().clip(0, 1)

inp_np = to_np((inp_sd + 1) / 2)
gen_np = to_np(gen)
gt_np  = to_np((gt + 1) / 2)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(inp_np); axes[0].set_title('Input Photo')
axes[1].imshow(gen_np); axes[1].set_title(f'Generated (CFG={GUIDANCE} str={STRENGTH})')
axes[2].imshow(gt_np);  axes[2].set_title('Ground Truth')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig('/content/sd_clip_result.png', dpi=150)
plt.show()

mse = ((gen - (gt + 1) / 2) ** 2).mean().item()
psnr = -10 * np.log10(mse) if mse > 0 else float('inf')
print(f'PSNR: {psnr:.2f} dB')